## 1. Load the Cleaned Dataset

The cleaned dataset generated during the data cleaning phase is loaded into a Pandas DataFrame.

This dataset will be used to select the features that are relevant for predicting delivery time.


In [8]:
import pandas as pd

df = pd.read_csv("../data/feature_engineered_data.csv")

## 2. Identify the Target Variable

`Time_taken(min)` is the target variable because it represents the delivery time we want to predict.

It will be separated from the input features before building the model.


In [9]:
y = df["Time_taken(min)"]

In [10]:
y.head()

0    24
1    33
2    26
3    21
4    30
Name: Time_taken(min), dtype: int64

## 3. Remove the Raw Date Feature

The original `Order_Date` column is removed from the feature set because useful information has already been extracted from it through `day_of_week`, `is_weekend`, and `month`.

The derived features will be used instead of the raw date.


In [12]:
df.columns

Index(['Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude',
       'Restaurant_longitude', 'Delivery_location_latitude',
       'Delivery_location_longitude', 'Order_Date', 'Weatherconditions',
       'Type_of_vehicle', 'Road_traffic_density', 'multiple_deliveries',
       'Time_taken(min)', 'distance', 'order_hour', 'pickup_hour',
       'day_of_week', 'is_weekend', 'month'],
      dtype='str')

## 3. Remove Non-Feature Columns

The target variable `Time_taken(min)` is excluded from the input features because it is the value we want to predict.

The raw `Order_Date` is also excluded because its useful information has already been transformed into `day_of_week`, `is_weekend`, and `month`.


In [13]:
X = df.drop(columns=["Time_taken(min)", "Order_Date"])

In [14]:
X.columns

Index(['Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude',
       'Restaurant_longitude', 'Delivery_location_latitude',
       'Delivery_location_longitude', 'Weatherconditions', 'Type_of_vehicle',
       'Road_traffic_density', 'multiple_deliveries', 'distance', 'order_hour',
       'pickup_hour', 'day_of_week', 'is_weekend', 'month'],
      dtype='str')

## 4. Check Redundant Numerical Features

Highly correlated features may contain similar information.

In the EDA phase, `order_hour` and `pickup_hour` showed a strong correlation. We will verify this relationship before deciding whether both features should be kept.


In [15]:
X[["order_hour", "pickup_hour"]].corr()

,order_hour,pickup_hour
order_hour,1.000000,0.792732
pickup_hour,0.792732,1.000000


## 5. Check Correlation Between Numerical Features

A correlation matrix is used to identify strongly related numerical features.

This helps detect possible redundancy between input features before training the model.


In [16]:
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns

corr = X[numerical_cols].corr()

corr

,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,multiple_deliveries,distance,order_hour,pickup_hour,is_weekend,month
Delivery_person_Age,1.000000,-0.113267,0.004314,0.009327,0.004308,0.009318,0.114197,-0.001016,0.001903,0.001111,0.004125,-0.005269
Delivery_person_Ratings,-0.113267,1.000000,-0.000382,0.003194,-0.001073,0.002113,-0.114941,-0.102785,-0.060085,-0.051093,-0.000817,-0.004953
Restaurant_latitude,0.004314,-0.000382,1.000000,0.001646,0.999977,0.002129,0.010295,0.018698,0.011115,0.012312,-0.009208,-0.225305
Restaurant_longitude,0.009327,0.003194,0.001646,1.000000,0.001644,0.999945,0.008534,-0.000941,0.001267,0.003051,-0.001801,-0.153731
Delivery_location_latitude,0.004308,-0.001073,0.999977,0.001644,1.000000,0.002197,0.011109,0.025417,0.014921,0.015596,-0.009209,-0.225384
Delivery_location_longitude,0.009318,0.002113,0.002129,0.999945,0.002197,1.000000,0.009811,0.009577,0.007225,0.008192,-0.001808,-0.153964
multiple_deliveries,0.114197,-0.114941,0.010295,0.008534,0.011109,0.009811,1.000000,0.121239,0.063324,0.073034,-0.005167,-0.009841
distance,-0.001016,-0.102785,0.018698,-0.000941,0.025417,0.009577,0.121239,1.000000,0.566825,0.489052,-0.000391,-0.015562
order_hour,0.001903,-0.060085,0.011115,0.001267,0.014921,0.007225,0.063324,0.566825,1.000000,0.792732,0.008979,-0.007968
pickup_hour,0.001111,-0.051093,0.012312,0.003051,0.015596,0.008192,0.073034,0.489052,0.792732,1.000000,0.002624,-0.007161


## 6. Remove Redundant Geographic Features

The dataset contains four raw coordinate features representing the restaurant and delivery locations.

Since `distance` already summarizes the geographic separation between the two locations, the raw coordinates are removed for the first model to keep the feature set simpler and reduce redundancy.


In [ ]:
X = X.drop(columns=[
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude"
])


In [20]:
X.columns

Index(['Delivery_person_Age', 'Delivery_person_Ratings', 'Weatherconditions',
       'Type_of_vehicle', 'Road_traffic_density', 'multiple_deliveries',
       'distance', 'order_hour', 'pickup_hour', 'day_of_week', 'is_weekend',
       'month'],
      dtype='str')

## 7. Check Related Date Features

`day_of_week` and `is_weekend` are related because `is_weekend` is derived from the day of the week.

Both features are kept initially because `day_of_week` provides detailed weekly information, while `is_weekend` provides a simpler weekday/weekend distinction.


In [21]:
pd.crosstab(X["day_of_week"], X["is_weekend"])

is_weekend,0,1
day_of_week,,
Friday,6325,0
Monday,5666,0
Saturday,0,5750
Sunday,0,5685
Thursday,5785,0
Tuesday,5812,0
Wednesday,6499,0


### 7.1 Remove Redundant Date Feature

During Feature Engineering, we created `day_of_week` and `is_weekend` from `Order_Date`.

The crosstab shows that `is_weekend` is completely determined by `day_of_week`: Monday to Friday always have `0`, while Saturday and Sunday always have `1`.

Therefore, `is_weekend` does not provide additional information when `day_of_week` is already present. We keep the more detailed `day_of_week` feature and remove `is_weekend`.


In [22]:
X = X.drop(columns=["is_weekend"])

### 8. Analyze `month`

The `month` feature was extracted from `Order_Date` during Feature Engineering.

Before deciding whether to keep it, we compare the average delivery time for each month to determine whether the month appears to contain useful information for the prediction.


In [23]:
df.groupby("month")["Time_taken(min)"].agg(["count", "mean", "median"])

,count,mean,median
month,,,
2,5819,26.567795,26.0
3,29773,26.302019,26.0
4,5930,26.135245,25.0


### 8.1 Month Feature Decision

The average delivery time is very similar across the three available months:

* February: 26.57 minutes
* March: 26.30 minutes
* April: 26.14 minutes

The differences are small, so `month` does not show a strong direct relationship with the target. However, we keep it for now because its effect may become useful when combined with other features such as weather or traffic.


### 9. Analyze `Weatherconditions`

`Weatherconditions` is a categorical feature that describes the weather during the delivery.

We first check the number of observations for each weather category to understand the distribution of this feature before deciding whether to keep it.


In [ ]:
df["Weatherconditions"].value_counts()

Weatherconditions
Fog           7382
Stormy        6932
Cloudy        6881
Sandstorms    6853
Windy         6792
Sunny         6682
Name: count, dtype: int64

In [25]:
df.groupby("Weatherconditions")["Time_taken(min)"].agg(["count", "mean", "median"])

,count,mean,median
Weatherconditions,,,
Cloudy,6881,28.931551,29.0
Fog,7382,28.844487,28.0
Sandstorms,6853,25.877718,26.0
Stormy,6932,25.892095,26.0
Sunny,6682,21.895091,20.0
Windy,6792,26.138840,26.0


### 9.1 Weather Feature Decision

The weather categories are reasonably well represented in the dataset. The average delivery time also varies noticeably between weather conditions, from about 21.9 minutes for Sunny weather to about 28.9 minutes for Cloudy weather.

Therefore, `Weatherconditions` appears to contain useful information for predicting delivery time, so we keep this feature.


In [26]:
X["Type_of_vehicle"].value_counts()

Type_of_vehicle
motorcycle           24197
scooter              13878
electric_scooter      3404
bicycle                 43
Name: count, dtype: int64

In [27]:
df.groupby("Type_of_vehicle")["Time_taken(min)"].agg(["count", "mean", "median"])

,count,mean,median
Type_of_vehicle,,,
bicycle,43,25.953488,26.0
electric_scooter,3404,24.469154,24.0
motorcycle,24197,27.611067,26.0
scooter,13878,24.510448,24.0


### 10 Vehicle Feature Decision

The average delivery time varies between vehicle types. Motorcycles have an average delivery time of about 27.6 minutes, while scooters and electric scooters are around 24.5 minutes.

Although `bicycle` has only 43 observations, `Type_of_vehicle` overall provides potentially useful information for predicting delivery time.

Therefore, we keep `Type_of_vehicle` as a feature.


In [28]:
X["Road_traffic_density"].value_counts()

Road_traffic_density
Low        14095
Jam        12950
Medium     10023
High        4046
Unknown      408
Name: count, dtype: int64

In [45]:
df["Road_traffic_density"].value_counts()

Road_traffic_density
Low        14503
Jam        12950
Medium     10023
High        4046
Name: count, dtype: int64

In [40]:
df.groupby("Road_traffic_density")["Time_taken(min)"].agg(["count", "mean", "median"])

,count,mean,median
Road_traffic_density,,,
High,4046,27.211567,27.0
Jam,12950,31.188571,31.0
Low,14503,21.423223,21.0
Medium,10023,26.736406,27.0


### 11 Delivery Person Age

We analyze `Delivery_person_Age` to see whether driver age has a meaningful relationship with delivery time.

We compare the number of observations, mean, and median delivery time for each age.


In [41]:
df.groupby("Delivery_person_Age")["Time_taken(min)"].agg(
    ["count", "mean", "median"]
)

,count,mean,median
Delivery_person_Age,,,
20.0,1955,22.941688,22.0
21.0,1961,22.953595,22.0
22.0,2021,22.958931,22.0
23.0,1933,23.269529,22.0
24.0,2035,22.998034,22.0
25.0,1984,22.903226,22.0
26.0,1980,22.967172,22.0
27.0,1966,23.067141,22.0
28.0,1997,23.170255,22.0


### 13 Delivery Person Ratings

We analyze `Delivery_person_Ratings` to determine whether driver ratings are associated with differences in delivery time.

We compare the number of observations, mean, and median delivery time for each rating.


In [42]:
df.groupby("Delivery_person_Ratings")["Time_taken(min)"].agg(
    ["count", "mean", "median"]
)

,count,mean,median
Delivery_person_Ratings,,,
1.0,25,25.600000,26.0
2.5,18,36.722222,35.0
2.6,20,38.800000,40.0
2.7,21,36.095238,37.0
2.8,17,36.529412,37.0
2.9,18,38.222222,39.0
3.0,6,32.666667,32.5
3.1,28,36.750000,35.0
3.2,26,36.576923,36.0


### 14 Multiple Deliveries

We analyze `multiple_deliveries` to determine whether the number of deliveries assigned to a driver is associated with delivery time.


In [43]:
df.groupby("multiple_deliveries")["Time_taken(min)"].agg(
    ["count", "mean", "median"]
)

,count,mean,median
multiple_deliveries,,,
0.0,12826,22.894121,22.0
1.0,26544,26.733499,26.0
2.0,1829,40.439038,40.0
3.0,323,47.842105,48.0


### 15 Distance

We analyze `distance` to determine whether the delivery distance is associated with delivery time.


In [44]:
df["distance_group"] = pd.cut(
    df["distance"],
    bins=[0, 5, 10, 15, 20, 25],
    labels=["0-5 km", "5-10 km", "10-15 km", "15-20 km", "20-25 km"]
)

df.groupby("distance_group", observed=True)["Time_taken(min)"].agg(
    ["count", "mean", "median"]
)

,count,mean,median
distance_group,,,
0-5 km,11187,22.184857,21.0
5-10 km,11187,24.389023,25.0
10-15 km,11183,29.841366,29.0
15-20 km,6186,29.897349,29.0
20-25 km,1779,29.784711,29.0


In [46]:
X.columns

Index(['Delivery_person_Age', 'Delivery_person_Ratings', 'Weatherconditions',
       'Type_of_vehicle', 'Road_traffic_density', 'multiple_deliveries',
       'distance', 'order_hour', 'pickup_hour', 'day_of_week', 'month'],
      dtype='str')

### 16 Analyze Order and Pickup Hours

`order_hour` represents the hour when the order was placed, while `pickup_hour` represents the hour when the order was picked up.

These two features are strongly correlated, so we analyze their relationship with the target before deciding whether both should be kept.


In [47]:
df.groupby("order_hour")["Time_taken(min)"].agg(
    ["count", "mean", "median"]
)

,count,mean,median
order_hour,,,
0.0,395,21.967089,21.0
8.0,1673,19.652720,19.0
9.0,1779,19.544688,19.0
10.0,1820,19.492308,19.0
11.0,1796,26.417038,27.0
12.0,823,26.849332,27.0
13.0,716,27.587989,27.0
14.0,721,27.327323,27.0
15.0,797,23.257215,23.0


In [48]:
df.groupby("pickup_hour")["Time_taken(min)"].agg(
    ["count", "mean", "median"]
)

,count,mean,median
pickup_hour,,,
0,1196,22.273411,21.0
8,1370,19.598540,19.0
9,1854,19.544229,19.0
10,1917,19.485655,19.0
11,1815,25.053444,25.0
12,1061,26.993402,27.0
13,728,27.611264,27.0
14,743,27.624495,27.0
15,824,23.822816,25.0


### 16.1 Order and Pickup Hours Decision

Both `order_hour` and `pickup_hour` show clear patterns with delivery time. Delivery times are generally higher during the evening, especially between 19:00 and 21:00.

Although the two features are strongly correlated, they represent different moments in the delivery process. Therefore, both features are kept for now and will be evaluated later during modeling.


### 17.1 Analyze Day of Week

`day_of_week` represents the day on which the order was placed.

We analyze the average and median delivery time for each day to determine whether this feature provides useful information for predicting delivery time.


In [49]:
pd.DataFrame({
    "day_of_week": X["day_of_week"],
    "target": y
}).groupby("day_of_week")["target"].agg(
    ["count", "mean", "median"]
)

,count,mean,median
day_of_week,,,
Friday,6325,26.846008,26.0
Monday,5666,26.237381,25.0
Saturday,5750,25.969043,25.0
Sunday,5685,26.457344,25.0
Thursday,5785,25.287640,25.0
Tuesday,5812,25.443221,25.0
Wednesday,6499,27.744422,27.0


### 18 Analyze Weather Conditions

`Weatherconditions` represents the weather condition during the delivery.

We analyze the average and median delivery time for each weather category to determine whether this feature provides useful information for predicting delivery time.


In [50]:
pd.DataFrame({
    "Weatherconditions": X["Weatherconditions"],
    "target": y
}).groupby("Weatherconditions")["target"].agg(
    ["count", "mean", "median"]
)

,count,mean,median
Weatherconditions,,,
Cloudy,6881,28.931551,29.0
Fog,7382,28.844487,28.0
Sandstorms,6853,25.877718,26.0
Stormy,6932,25.892095,26.0
Sunny,6682,21.895091,20.0
Windy,6792,26.138840,26.0


### 18.1 Weather Feature Decision

The average delivery time varies noticeably between weather conditions. Sunny weather has the lowest average delivery time at about 21.9 minutes, while Cloudy and Foggy conditions have averages close to 28.9 minutes.

The categories also have a reasonable number of observations, so the results are representative of the dataset.

Therefore, `Weatherconditions` contains useful information for predicting delivery time and will be kept as a feature.


### 19 Analyze Vehicle Type

`Type_of_vehicle` represents the type of vehicle used for the delivery.

We analyze the average and median delivery time for each vehicle type to determine whether this feature provides useful information for predicting delivery time.


In [51]:
pd.DataFrame({
    "Type_of_vehicle": X["Type_of_vehicle"],
    "target": y
}).groupby("Type_of_vehicle")["target"].agg(
    ["count", "mean", "median"]
)

,count,mean,median
Type_of_vehicle,,,
bicycle,43,25.953488,26.0
electric_scooter,3404,24.469154,24.0
motorcycle,24197,27.611067,26.0
scooter,13878,24.510448,24.0


### 19.1 Vehicle Type Feature Decision

The average delivery time varies between vehicle types. Motorcycles have an average delivery time of about 27.6 minutes, while scooters and electric scooters have averages around 24.5 minutes.

The `bicycle` category contains only 43 observations, so its individual result should be interpreted carefully. However, `Type_of_vehicle` overall provides useful information about delivery time.

Therefore, `Type_of_vehicle` will be kept as a feature.


In [53]:

print(X.columns.tolist())

['Delivery_person_Age', 'Delivery_person_Ratings', 'Weatherconditions', 'Type_of_vehicle', 'Road_traffic_density', 'multiple_deliveries', 'distance', 'order_hour', 'pickup_hour', 'day_of_week', 'month']


In [58]:
X.shape
X.columns
y.head()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (41522, 11)
y shape: (41522,)


## Conclusion

In this notebook, we separated the target variable from the input features and analyzed the available features using correlation and target-based comparisons.

We removed redundant features such as the raw coordinates and `is_weekend`, while keeping features that may provide useful information for predicting delivery time.

The final dataset contains **11 selected features** and **41,522 observations**. These features will now be prepared for machine learning in the preprocessing step.
